# BRISC Medical Image Segmentation with U-Net
This notebook follows the same lab-style flow: imports, config, dataset class, visualisation, model, loss, metrics, training, validation, and checkpoint saving.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import InterpolationMode
from torchvision.transforms import functional as TF

try:
    import wandb
except ImportError:
    wandb = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Configure Model Hyperparameters and Locations


In [ ]:
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/INM705-CW"
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/INM705-CW/data/segmentation_task"
IMAGE_SIZE = 256
BATCH_SIZE = 8
NUM_WORKERS = 2
EPOCHS = 25
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
BASE_CHANNELS = 64
USE_AUGMENTATION = False
SEED = 42

USE_WANDB = True
WANDB_PROJECT = "inm705-medical-segmentation"
WANDB_RUN_NAME = "brisc-unet-no-augmentation"

CHECKPOINT_DIR = "/content/drive/MyDrive/Colab Notebooks/INM705-CW/checkpoints"
SAMPLES_DIR = "/content/drive/MyDrive/Colab Notebooks/INM705-CW/samples"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)

## Set Random Seed for Reproducibility


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

## Dataset and dataloaders


In [ ]:
VALID_EXTENSIONS = {".png", ".jpg"}

def list_image_files(folder):
    files = [p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS]
    return sorted(files)

def match_image_mask_pairs(images_dir, masks_dir):
    image_files = list_image_files(images_dir)
    if len(image_files) == 0:
        raise ValueError(f"No image files found in: {images_dir}")

    mask_by_stem = {}
    for mask_path in list_image_files(masks_dir):
        mask_by_stem[mask_path.stem] = mask_path

    pairs = []
    for image_path in image_files:
        mask_path = mask_by_stem.get(image_path.stem)
        if mask_path is None:
            raise FileNotFoundError(f"Mask file not found for image stem '{image_path.stem}' in {masks_dir}")
        pairs.append((image_path, mask_path))
    return pairs

class SegmentationDataset(Dataset):
    def __init__(self, pairs, image_size, train, use_augmentation=True):
        self.pairs = list(pairs)
        self.image_size = image_size
        self.train = train
        self.use_augmentation = use_augmentation

    def __len__(self):
        return len(self.pairs)

    def augment(self, image, mask):
        if random.random() < 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)
        if random.random() < 0.5:
            image = TF.vflip(image)
            mask = TF.vflip(mask)
        if random.random() < 0.2:
            angle = random.uniform(-20.0, 20.0)
            image = TF.rotate(image, angle, interpolation=InterpolationMode.BILINEAR)
            mask = TF.rotate(mask, angle, interpolation=InterpolationMode.NEAREST)
        return image, mask

    def __getitem__(self, idx):
        image_path, mask_path = self.pairs[idx]

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        image = TF.resize(image, [self.image_size, self.image_size], interpolation=InterpolationMode.BILINEAR)
        mask = TF.resize(mask, [self.image_size, self.image_size], interpolation=InterpolationMode.NEAREST)

        if self.train and self.use_augmentation:
          image, mask = self._augment(image, mask)

        image = TF.to_tensor(image)
        image = TF.normalize(image, mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])

        mask = TF.to_tensor(mask)
        mask = (mask > 0.5).float()

        return image, mask

def build_brisc_dataloaders(data_dir, image_size, batch_size, num_workers, use_augmentation=True):
    root = Path(data_dir)
    seg_root = root / "segmentation_task"

    train_images_dir = root / "train" / "images"
    train_masks_dir = root / "train" / "masks"
    test_images_dir = root / "test" / "images"
    test_masks_dir = root / "test" / "masks"

    required_dirs = [train_images_dir, train_masks_dir, test_images_dir, test_masks_dir]
    if any(not d.is_dir() for d in required_dirs):
        raise FileNotFoundError(
            "Expected BRISC structure: data_dir/segmentation_task/train/{images,masks} "
            "and data_dir/segmentation_task/test/{images,masks}"
        )

    train_pairs = match_image_mask_pairs(train_images_dir, train_masks_dir)
    val_pairs = match_image_mask_pairs(test_images_dir, test_masks_dir)

    train_dataset = SegmentationDataset(train_pairs, image_size=image_size, train=True, use_augmentation=use_augmentation)
    val_dataset = SegmentationDataset(val_pairs, image_size=image_size, train=False, use_augmentation=False)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )

    return train_loader, val_loader, train_dataset, val_dataset

## Create dataloaders

In [ ]:
train_loader, val_loader, train_dataset, val_dataset = build_brisc_dataloaders(
    data_dir=DATA_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    use_augmentation = USE_AUGMENTATION
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

## Visualise Samples


In [ ]:
def denormalize_image(image):
    image = (image * 0.5) + 0.5
    return torch.clamp(image, 0.0, 1.0)

fig, axes = plt.subplots(2, 10, figsize=(12, 8))
for i in range(10):
    image, mask = train_dataset[i]
    axes[0, i].imshow(denormalize_image(image).permute(1, 2, 0))
    axes[0, i].set_title("Image")
    axes[0, i].axis("off")

    axes[1, i].imshow(mask.squeeze(0), cmap="gray")
    axes[1, i].set_title("Mask")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

## U-Net model

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, base_channels=32):
        super().__init__()

        c1 = base_channels
        c2 = base_channels * 2
        c3 = base_channels * 4
        c4 = base_channels * 8

        self.enc1 = DoubleConv(in_channels, c1)
        self.enc2 = DoubleConv(c1, c2)
        self.enc3 = DoubleConv(c2, c3)
        self.enc4 = DoubleConv(c3, c4)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(c4, c4 * 2)

        self.up4 = nn.ConvTranspose2d(c4 * 2, c4, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(c4 * 2, c4)

        self.up3 = nn.ConvTranspose2d(c4, c3, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(c3 * 2, c3)

        self.up2 = nn.ConvTranspose2d(c3, c2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(c2 * 2, c2)

        self.up1 = nn.ConvTranspose2d(c2, c1, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(c1 * 2, c1)

        self.out_conv = nn.Conv2d(c1, out_channels, kernel_size=1)

    def resize_to_match(self, src, target):
        if src.shape[-2:] == target.shape[-2:]:
            return src
        return F.interpolate(src, size=target.shape[-2:], mode="bilinear", align_corners=False)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([self.resize_to_match(d4, e4), e4], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([self.resize_to_match(d3, e3), e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([self.resize_to_match(d2, e2), e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([self.resize_to_match(d1, e1), e1], dim=1))

        return self.out_conv(d1)

model = UNet(in_channels=3, out_channels=1, base_channels=BASE_CHANNELS).to(device)
model

## Loss and metrics

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(probs.size(0), -1)
        targets = targets.view(targets.size(0), -1)

        intersection = (probs * targets).sum(dim=1)
        denominator = probs.sum(dim=1) + targets.sum(dim=1)
        dice = (2.0 * intersection + self.smooth) / (denominator + self.smooth)
        return 1.0 - dice.mean()

class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight

    def forward(self, logits, targets):
        bce = self.bce(logits, targets)
        dice = self.dice(logits, targets)
        return self.bce_weight * bce + (1.0 - self.bce_weight) * dice

@torch.no_grad()
def binary_dice_score(logits, targets, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)
    denominator = preds.sum(dim=1) + targets.sum(dim=1)
    dice = (2.0 * intersection + eps) / (denominator + eps)
    return dice.mean()

@torch.no_grad()
def binary_iou_score(logits, targets, threshold=0.5, eps=1e-6):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    preds = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean()

criterion = BCEDiceLoss(bce_weight=0.5)

## Optimiser and scheduler

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)

## Helper functions

In [ ]:
def save_prediction_overlay(image, mask_gt, mask_pred, output_path):
    image_np = denormalize_image(image).permute(1, 2, 0).cpu().numpy()
    gt_np = (mask_gt.squeeze(0).cpu().numpy() > 0.5).astype(np.uint8)
    pred_np = (mask_pred.squeeze(0).cpu().numpy() > 0.5).astype(np.uint8)

    overlay = image_np.copy()
    overlay[gt_np == 1, 1] = np.clip(overlay[gt_np == 1, 1] + 0.5, 0, 1)
    overlay[pred_np == 1, 0] = np.clip(overlay[pred_np == 1, 0] + 0.5, 0, 1)

    out = (overlay * 255).astype(np.uint8)
    Image.fromarray(out).save(output_path)

def evaluate(model, loader, criterion, device):
    model.eval()
    losses = []
    dices = []
    ious = []

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            logits = model(images)

            loss = criterion(logits, masks)
            dice = binary_dice_score(logits, masks)
            iou = binary_iou_score(logits, masks)

            losses.append(loss.item())
            dices.append(dice.item())
            ious.append(iou.item())

    return (
        sum(losses) / max(1, len(losses)),
        sum(dices) / max(1, len(dices)),
        sum(ious) / max(1, len(ious)),
    )

## WandB initialisation

In [ ]:
import wandb
wandb.login()

In [ ]:
use_wandb = USE_WANDB and (wandb is not None)

if use_wandb:
    wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        config={
            "data_dir": DATA_DIR,
            "image_size": IMAGE_SIZE,
            "batch_size": BATCH_SIZE,
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "base_channels": BASE_CHANNELS,
            "seed": SEED,
            "train_size": len(train_dataset),
            "val_size": len(val_dataset),
        }
    )
elif USE_WANDB and wandb is None:
    print("wandb is not installed, so training will continue without logging.")

## Training loop

In [ ]:
best_dice = -1.0
history = {
    "train_loss": [],
    "val_loss": [],
    "val_dice": [],
    "val_iou": []
}

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_losses = []

    for images, masks in train_loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = criterion(logits, masks)

        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())

    train_loss = sum(epoch_losses) / max(1, len(epoch_losses))
    val_loss, val_dice, val_iou = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_dice)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_dice"].append(val_dice)
    history["val_iou"].append(val_iou)

    if use_wandb:
        wandb.log({
            "epoch": epoch,
            "train/loss": train_loss,
            "val/loss": val_loss,
            "val/dice": val_dice,
            "val/iou": val_iou,
            "lr": optimizer.param_groups[0]["lr"],
        })

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "image_size": IMAGE_SIZE,
                "base_channels": BASE_CHANNELS,
                "best_dice": best_dice,
            },
            os.path.join(CHECKPOINT_DIR, "unet_best.pt")
        )

    with torch.no_grad():
        sample_images, sample_masks = next(iter(val_loader))
        sample_images = sample_images.to(device)
        logits = model(sample_images)
        preds = (torch.sigmoid(logits) > 0.5).float().cpu()

        sample_path = os.path.join(SAMPLES_DIR, f"epoch_{epoch:03d}_overlay.png")
        save_prediction_overlay(sample_images[0].cpu(), sample_masks[0], preds[0], sample_path)

        if use_wandb:
            wandb.log({"val/overlay": wandb.Image(sample_path), "epoch": epoch})

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_dice={val_dice:.4f} | "
        f"val_iou={val_iou:.4f}"
    )

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "image_size": IMAGE_SIZE,
        "base_channels": BASE_CHANNELS,
        "best_dice": best_dice,
    },
    os.path.join(CHECKPOINT_DIR, "unet_final.pt")
)

if use_wandb:
    wandb.finish()

## Plot training curves

In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.plot(epochs_range, history["train_loss"], label="Train loss")
plt.plot(epochs_range, history["val_loss"], label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss curves")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs_range, history["val_dice"], label="Val Dice")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.title("Validation Dice")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(epochs_range, history["val_iou"], label="Val IoU")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.title("Validation IoU")
plt.legend()

plt.tight_layout()
plt.show()

## Visualise predictions

In [ ]:
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, "unet_test2_best.pt"), map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

images, masks = next(iter(val_loader))
images = images.to(device)

with torch.no_grad():
    logits = model(images)
    preds = (torch.sigmoid(logits) > 0.5).float().cpu()

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i in range(3):
    axes[i, 0].imshow(denormalize_image(images[i].cpu()).permute(1, 2, 0))
    axes[i, 0].set_title("Image")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(masks[i].squeeze(0), cmap="gray")
    axes[i, 1].set_title("Ground truth")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(preds[i].squeeze(0), cmap="gray")
    axes[i, 2].set_title("Prediction")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()